In [ ]:
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field
from google import genai #type: ignore
from google.genai import types #type: ignore
from google.colab import drive #type: ignore
from dotenv import load_dotenv
import os
import asyncio
from enum import Enum

drive.mount("/content/drive")
load_dotenv("/content/drive/MyDrive/.env")
api_key = os.getenv("GOOGLE_API_KEY")  

In [ ]:
client = genai.Client(api_key=api_key)
MODEL_ID = "gemini-2.5-flash"

In [ ]:
DOCUMENT = """
# Q3 2023 Financial Performance Analysis

The Q3 earnings report shows a 20% increase in revenue and a 15% growth in user engagement, 
beating market expectations. These impressive results reflect our successful product strategy 
and strong market positioning.

Our core business segments demonstrated remarkable resilience, with digital services leading 
the growth at 25% year-over-year. The expansion into new markets has proven particularly 
successful, contributing to 30% of the total revenue increase.

Customer acquisition costs decreased by 10% while retention rates improved to 92%, 
marking our best performance to date. These metrics, combined with our healthy cash flow 
position, provide a strong foundation for continued growth into Q4 and beyond.
"""

In [ ]:
def search_google_drive(query: str) -> dict:
    """
    Searches for a file on Google Drive and returns its content or a summary.

    Args:
        query (str): The search query to find the file, e.g., 'Q3 earnings report'.

    Returns:
        dict: A dictionary representing the search results, including file names and summaries.
    """
    
    return {
        "files": [
            {
                "name": "Q3_Earnings_Report_2024.pdf",
                "id": "file12345",
                "content": DOCUMENT,
            }
        ]
    }

In [ ]:
def send_discord_message(channel_id: str, message: str) -> dict:
    """
    Sends a message to a specific Discord channel.

    Args:
        channel_id (str): The ID of the channel to send the message to, e.g., '#finance'.
        message (str): The content of the message to send.

    Returns:
        dict: A dictionary confirming the action, e.g., {"status": "success"}.
    """
    
    return {
        "status": "success",
        "status_code": 200,
        "channel": channel_id,
        "message_preview": f"{message[:50]}...",
    }

In [ ]:
def summarize_financial_report(text: str) -> str:
    """
    Summarizes a financial report.

    Args:
        text (str): The text to summarize.

    Returns:
        str: The summary of the text.
    """
    
    return "The Q3 2023 earnings report shows strong performance across all metrics..."


In [ ]:
def summarize_financial_report(text: str) -> str:
    """
    Summarizes a financial report.

    Args:
        text (str): The text to summarize.

    Returns:
        str: The summary of the text.
    """
    
    return "The Q3 2023 earnings report shows strong performance across all metrics..."


In [ ]:
send_discord_message_schema = {
    "name": "send_discord_message",
    "description": "Sends a message to a specific Discord channel.",
    "parameters": {
        "type": "object",
        "properties": {
            "channel_id": {
                "type": "string",
                "description": "The ID of the channel to send the message to, e.g., '#finance'."
            },
            "message": {
                "type": "string",
                "description": "The content of the message to send."
            }
        },
        "required": ["channel_id", "message"]
    }
}

In [ ]:
summarize_financial_report_schema = {
    "name": "summarize_financial_report",
    "description": "Summarizes a financial report.",
    "parameters": {
        "type": "object",
        "properties": {
            "text": {
                "type": "string",
                "description": "The text to summarize."
            }
        },
        "required": ["text"]
    }
}

In [ ]:
TOOLS = {
    "search_google_drive": {
        "handler": search_google_drive,
        "schema": search_google_drive_schema,
    },
    "send_discord_message": {
        "handler": send_discord_message,
        "schema": send_discord_message_schema,
    },
    "summarize_financial_report": {
        "handler": summarize_financial_report,
        "schema": summarize_financial_report_schema,
    },
}
TOOLS_BY_NAME = {
		tool_name: tool["handler"] for tool_name, tool in TOOLS.items()
}
TOOLS_SCHEMA = [tool["schema"] for tool in TOOLS.values()]

In [ ]:
TOOLS = {
    "search_google_drive": {
        "handler": search_google_drive,
        "schema": search_google_drive_schema,
    },
    "send_discord_message": {
        "handler": send_discord_message,
        "schema": send_discord_message_schema,
    },
    "summarize_financial_report": {
        "handler": summarize_financial_report,
        "schema": summarize_financial_report_schema,
    },
}
TOOLS_BY_NAME = {
		tool_name: tool["handler"] for tool_name, tool in TOOLS.items()
}
TOOLS_SCHEMA = [tool["schema"] for tool in TOOLS.values()]

In [ ]:
USER_PROMPT = "Can you help me find the latest quarterly report and share key insights with the team?"

messages = [
		TOOL_CALLING_SYSTEM_PROMPT.format(tools=str(TOOLS_SCHEMA)),
		USER_PROMPT
]

response = client.models.generate_content(
    model=MODEL_ID,
    contents=messages,
)

In [ ]:
def extract_tool_call(response_text: str) -> str:
    """
    Extracts the tool call from the response text.
    """
    return response_text.split("<tool_call>")[1].split("</tool_call>")[0].strip()

tool_call_str = extract_tool_call(response.text)

import json

tool_call = json.loads(tool_call_str)

tool_handler = TOOLS_BY_NAME[tool_call["name"]]

tool_result = tool_handler(**tool_call["args"])

In [ ]:
def call_tool(response_text: str, tools_by_name: dict):
    tool_call_str = extract_tool_call(response_text)
    tool_call = json.loads(tool_call_str)
    tool_name = tool_call["name"]
    tool_args = tool_call["args"]
    tool = tools_by_name[tool_name]
    return tool(**tool_args)

# Using the helper function
call_tool(response.text, tools_by_name=TOOLS_BY_NAME)

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=f"Interpret the tool result: {json.dumps(tool_result, indent=2)}",
)